# Phase 4 — SHAP Explainability

**MSc Credit Risk Project — Colab handover (4 of 6)**

Fits the champion XGBoost (TreeSHAP) and the logistic baseline (LinearSHAP) on the
full training window, then computes:

1. Feature importance tables (top 30, both models)
2. Beeswarm plot (TreeSHAP)
3. Dependence panels for the top features
4. **Three worked case studies** (a true positive, a true negative, a false positive)
   with the bar-chart attribution panels from Chapter 5 of the dissertation
5. Tree-vs-linear top-30 overlap analysis

**Checkpoints:** `reports/tables/shap_*.csv`, `reports/figures/shap_*.png`,
`models/champion_xgboost.joblib`. **Runtime:** ~5 minutes.


In [ ]:
# ============================================================
# SETUP: mount Google Drive and locate the project folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# AUTO-DETECT the project root (folder containing src/ and run_experiment.py).
# If auto-detection fails, set PROJECT_ROOT manually, e.g.:
#   PROJECT_ROOT = Path('/content/drive/MyDrive/credit-risk-project - Copy')
PROJECT_ROOT = None
for candidate in Path('/content/drive/MyDrive').rglob('run_experiment.py'):
    if (candidate.parent / 'src').is_dir():
        PROJECT_ROOT = candidate.parent
        break
assert PROJECT_ROOT is not None, "Could not find the project folder on Drive - set PROJECT_ROOT manually"

os.chdir(PROJECT_ROOT)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
# Ensure expected directories exist (log files are opened at import time)
for _d in ('logs', 'data/raw', 'data/processed', 'models',
           'reports/tables', 'reports/figures', 'outputs/eda'):
    os.makedirs(PROJECT_ROOT / _d, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print("Working directory set. All outputs are saved here (persistent on Drive).")


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from src.config import TABLES_DIR

if (TABLES_DIR / "shap_importance_top30.csv").exists():
    print("SHAP checkpoint found - SKIPPING SHAP pipeline.")
else:
    from src.explainability.shap_engine import run_shap_pipeline
    run_shap_pipeline()
    print("SHAP pipeline complete.")


In [ ]:
# ---- Display SHAP results inline ----
import pandas as pd
from IPython.display import Image, display
from src.config import FIGURES_DIR

imp = pd.read_csv(TABLES_DIR / "shap_importance_top30.csv")
print("Top 15 features by mean |SHAP| (XGBoost / TreeSHAP):")
display(imp.head(15).round(4))

for name in ["shap_importance.png", "shap_beeswarm.png",
             "shap_dependence.png", "shap_case_studies.png"]:
    p = FIGURES_DIR / name
    if p.exists():
        display(Image(str(p), width=800))


In [ ]:
# Save the trained champion model as a shareable artefact (optional handover extra)
import joblib
import numpy as np
from src.config import MODELS_DIR, RANDOM_SEED
from src.models.factory import make_model
from src.explainability.shap_engine import load_training_matrix

X, y, ids, feature_names = load_training_matrix()
spw = float(np.sum(y == 0)) / max(float(np.sum(y == 1)), 1.0)
champion = make_model("xgboost", "baseline", spw, RANDOM_SEED)
champion.fit(X, y)
joblib.dump({"model": champion, "feature_names": feature_names},
            MODELS_DIR / "champion_xgboost.joblib")
print(f"Champion XGBoost saved: {MODELS_DIR / 'champion_xgboost.joblib'}")


In [ ]:
# ---- Phase 4 verification against dissertation Chapter 5 ----
imp = pd.read_csv(TABLES_DIR / "shap_importance_top30.csv")
assert imp.iloc[0]['feature'] == 'EXT_SOURCES_MEAN'
assert abs(imp.iloc[0]['mean_abs_shap'] - 0.438) < 0.02

cases = pd.read_csv(TABLES_DIR / "shap_case_studies.csv")
assert cases['applicant_id'].unique().tolist() == [187933, 237456, 334854]

with open(TABLES_DIR / "shap_tree_vs_linear_overlap.txt") as f:
    assert "16/30" in f.read()
print("PHASE 4 CHECKPOINT OK: EXT_SOURCES_MEAN top (0.438), cases 187933/237456/334854, overlap 16/30")
